# Repo Architecture Walkthrough

This notebook orients a new contributor to the repository before they dive into any implementation details.

By the end, you should know:

- where the runtime code lives
- which files make up the public entrypoints
- which docs describe the live system today
- which notebook to open next


In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root. Start Jupyter from somewhere inside the repo.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
FIXTURES_DIR = REPO_ROOT / "tests" / "fixtures"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


def reset_dir(path: Path) -> Path:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def copy_fixture_bundle(destination: Path, fixture_name: str, *filenames: str) -> dict[str, Path]:
    destination.mkdir(parents=True, exist_ok=True)
    copied: dict[str, Path] = {}
    for filename in filenames:
        source = FIXTURES_DIR / fixture_name / filename
        target = destination / filename
        shutil.copyfile(source, target)
        copied[filename] = target
    return copied

import tomllib

WORKDIR = reset_dir(REPO_ROOT / "var" / "walkthrough" / "00_repo_architecture")
with (REPO_ROOT / "pyproject.toml").open("rb") as handle:
    pyproject = tomllib.load(handle)

print(f"Repo root: {REPO_ROOT}")
print(f"Scratch dir: {WORKDIR}")
print("Console entrypoints:")
for name, target in pyproject["project"]["scripts"].items():
    print(f"  - {name}: {target}")


## Top-Level Repo Map

The repo keeps runtime code, docs, examples, notebooks, tests, and generated local state in clearly separate places.


In [ ]:
important_paths = [
    "README.md",
    "docs",
    "examples",
    "notebooks",
    "src/mtg_source_stack",
    "tests",
    "var",
]
for relative in important_paths:
    path = REPO_ROOT / relative
    kind = "dir" if path.is_dir() else "file"
    print(f"{kind:>4}  {relative}")


## Runtime Package Boundaries

The active package is `src/mtg_source_stack/`. The repo is intentionally organized around four main runtime areas: `cli/`, `db/`, `importer/`, and `inventory/`.


In [ ]:
runtime_root = REPO_ROOT / "src" / "mtg_source_stack"
for path in sorted(runtime_root.iterdir()):
    if path.name == "__pycache__":
        continue
    kind = "dir" if path.is_dir() else "file"
    print(f"{kind:>4}  {path.relative_to(REPO_ROOT)}")


## Docs and Reading Order

The repo now has a docs landing page and a split notebook series. The current runtime story lives in the docs below; future design notes are separated out so they do not get mistaken for implemented behavior.


In [ ]:
docs_to_read = [
    "docs/README.md",
    "docs/architecture.md",
    "docs/backend_v1_contract.md",
    "docs/ingestion_flow.md",
    "docs/api_v1_contract.md",
]
for relative in docs_to_read:
    first_line = (REPO_ROOT / relative).read_text(encoding="utf-8").splitlines()[0]
    print(f"{relative}: {first_line}")


## What To Open Next

If you want to understand the live system in order, continue with:

1. `01_db_and_migrations_walkthrough.ipynb`
2. `02_importer_walkthrough.ipynb`
3. `03_inventory_domain_walkthrough.ipynb`
4. `04_reporting_and_api_contract_walkthrough.ipynb`
